In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import os

# Set your processed folder path
project_path = '/content/drive/MyDrive/Brazilian_Ecommerce'
processed_path = os.path.join(project_path, 'processed')

# Load all nine cleaned datasets
customers = pd.read_csv(os.path.join(processed_path, 'customers_clean.csv'))
geolocation = pd.read_csv(os.path.join(processed_path, 'geolocation_clean.csv'))
order_items = pd.read_csv(os.path.join(processed_path, 'order_items_clean.csv'))
order_payments = pd.read_csv(os.path.join(processed_path, 'order_payments_clean.csv'))
order_reviews = pd.read_csv(os.path.join(processed_path, 'order_reviews_clean.csv'))
orders = pd.read_csv(os.path.join(processed_path, 'orders_clean.csv'))
products = pd.read_csv(os.path.join(processed_path, 'products_clean.csv'))
sellers = pd.read_csv(os.path.join(processed_path, 'sellers_clean.csv'))
category_translation = pd.read_csv(os.path.join(processed_path, 'category_translation_clean.csv'))

print("All nine datasets loaded successfully!")

All nine datasets loaded successfully!


In [ ]:
datasets = {
    'customers': customers,
    'geolocation': geolocation,
    'order_items': order_items,
    'order_payments': order_payments,
    'order_reviews': order_reviews,
    'orders': orders,
    'products': products,
    'sellers': sellers,
    'category_translation': category_translation
}

# **Key Checks**

In [ ]:
key_checks = {
    'customers': ['customer_id', 'customer_unique_id'],
    'orders': ['order_id'],
    'order_items': ['order_id', 'order_item_id'],
    'order_payments': ['order_id', 'payment_sequential'],
    'order_reviews': ['review_id'],
    'products': ['product_id'],
    'sellers': ['seller_id']
}

for table, columns in key_checks.items():
    print(f"\n{table.upper()}")

    for column in columns:
        print(
            f"{column}: "
            f"{datasets[table][column].nunique():,} unique values "
            f"out of {len(datasets[table]):,} rows"
        )


CUSTOMERS
customer_id: 99,441 unique values out of 99,441 rows
customer_unique_id: 96,096 unique values out of 99,441 rows

ORDERS
order_id: 99,441 unique values out of 99,441 rows

ORDER_ITEMS
order_id: 98,666 unique values out of 112,650 rows
order_item_id: 21 unique values out of 112,650 rows

ORDER_PAYMENTS
order_id: 99,440 unique values out of 103,886 rows
payment_sequential: 29 unique values out of 103,886 rows

ORDER_REVIEWS
review_id: 98,410 unique values out of 99,224 rows

PRODUCTS
product_id: 32,951 unique values out of 32,951 rows

SELLERS
seller_id: 3,095 unique values out of 3,095 rows


# **Composite Key**

In [ ]:
composite_key_checks = {
    'order_items': ['order_id', 'order_item_id'],
    'order_payments': ['order_id', 'payment_sequential'],
    'order_reviews': ['review_id', 'order_id']
}

for table, columns in composite_key_checks.items():

    df = datasets[table]

    duplicate_count = df.duplicated(
        subset=columns,
        keep=False
    ).sum()

    null_count = df[columns].isnull().any(axis=1).sum()

    print(f"\n{table.upper()}")
    print("Columns checked:", columns)
    print("Rows involved in duplicate key combinations:", duplicate_count)
    print("Rows with missing key values:", null_count)


ORDER_ITEMS
Columns checked: ['order_id', 'order_item_id']
Rows involved in duplicate key combinations: 0
Rows with missing key values: 0

ORDER_PAYMENTS
Columns checked: ['order_id', 'payment_sequential']
Rows involved in duplicate key combinations: 0
Rows with missing key values: 0

ORDER_REVIEWS
Columns checked: ['review_id', 'order_id']
Rows involved in duplicate key combinations: 0
Rows with missing key values: 0


# **Remaining Tables**

In [ ]:
# Check geolocation key candidates

print("GEOLOCATION")

print(
    "Unique ZIP prefixes:",
    geolocation['geolocation_zip_code_prefix'].nunique()
)

print(
    "Duplicate ZIP + latitude + longitude combinations:",
    geolocation.duplicated(
        subset=[
            'geolocation_zip_code_prefix',
            'geolocation_lat',
            'geolocation_lng'
        ]
    ).sum()
)

print(
    "Missing values in ZIP prefix:",
    geolocation['geolocation_zip_code_prefix'].isna().sum()
)


# Check category translation

print("\nCATEGORY TRANSLATION")

print(
    "Unique Portuguese categories:",
    category_translation['product_category_name'].nunique()
)

print(
    "Duplicate Portuguese categories:",
    category_translation.duplicated(
        subset=['product_category_name']
    ).sum()
)

print(
    "Missing Portuguese category names:",
    category_translation['product_category_name'].isna().sum()
)

GEOLOCATION
Unique ZIP prefixes: 19015
Duplicate ZIP + latitude + longitude combinations: 18178
Missing values in ZIP prefix: 0

CATEGORY TRANSLATION
Unique Portuguese categories: 71
Duplicate Portuguese categories: 0
Missing Portuguese category names: 0


# **Cardinality Check**

In [ ]:
relationships = [
    ('customers', 'customer_id', 'orders', 'customer_id'),
    ('orders', 'order_id', 'order_items', 'order_id'),
    ('products', 'product_id', 'order_items', 'product_id'),
    ('sellers', 'seller_id', 'order_items', 'seller_id'),
    ('orders', 'order_id', 'order_payments', 'order_id'),
    ('orders', 'order_id', 'order_reviews', 'order_id')
]

for parent, parent_key, child, child_key in relationships:

    parent_values = set(datasets[parent][parent_key].dropna())
    child_values = set(datasets[child][child_key].dropna())

    unmatched = child_values - parent_values

    print(f"\n{parent} → {child}")
    print("Unmatched child values:", len(unmatched))


customers → orders
Unmatched child values: 0

orders → order_items
Unmatched child values: 0

products → order_items
Unmatched child values: 0

sellers → order_items
Unmatched child values: 0

orders → order_payments
Unmatched child values: 0

orders → order_reviews
Unmatched child values: 0


# **Relationship Check**

In [ ]:
relationship_checks = [
    ('customers', 'customer_id', 'orders', 'customer_id'),
    ('orders', 'order_id', 'order_items', 'order_id'),
    ('products', 'product_id', 'order_items', 'product_id'),
    ('sellers', 'seller_id', 'order_items', 'seller_id'),
    ('orders', 'order_id', 'order_payments', 'order_id'),
    ('orders', 'order_id', 'order_reviews', 'order_id')
]

for parent, parent_key, child, child_key in relationship_checks:

    counts = datasets[child].groupby(child_key).size()

    print(f"\n{parent.upper()} → {child.upper()}")
    print("Parent records with multiple child records:", (counts > 1).sum())
    print("Maximum child records for one parent:", counts.max())


CUSTOMERS → ORDERS
Parent records with multiple child records: 0
Maximum child records for one parent: 1

ORDERS → ORDER_ITEMS
Parent records with multiple child records: 9803
Maximum child records for one parent: 21

PRODUCTS → ORDER_ITEMS
Parent records with multiple child records: 14834
Maximum child records for one parent: 527

SELLERS → ORDER_ITEMS
Parent records with multiple child records: 2586
Maximum child records for one parent: 2033

ORDERS → ORDER_PAYMENTS
Parent records with multiple child records: 2961
Maximum child records for one parent: 29

ORDERS → ORDER_REVIEWS
Parent records with multiple child records: 547
Maximum child records for one parent: 3


# **Count orders per Customer**

In [ ]:
# Count orders per real customer

customer_order_counts = (
    customers.groupby('customer_unique_id')['customer_id']
    .count()
)

print("Total unique customers:",
      customers['customer_unique_id'].nunique())

print("Customers with more than one order:",
      (customer_order_counts > 1).sum())

print("Maximum orders by one customer:",
      customer_order_counts.max())

Total unique customers: 96096
Customers with more than one order: 2997
Maximum orders by one customer: 17


In [ ]:
# Create a mapping between customer_id and customer_unique_id

customer_mapping = customers[
    ['customer_id', 'customer_unique_id']
].drop_duplicates()

print("Customer mapping rows:", len(customer_mapping))
print("Unique customer IDs:", customer_mapping['customer_id'].nunique())
print("Unique real customer IDs:", customer_mapping['customer_unique_id'].nunique())

# Check whether every order can find its customer
orders_with_customer = orders.merge(
    customer_mapping,
    on='customer_id',
    how='left',
    indicator=True
)

print(
    "Orders without a customer match:",
    (orders_with_customer['_merge'] == 'left_only').sum()
)

Customer mapping rows: 99441
Unique customer IDs: 99441
Unique real customer IDs: 96096
Orders without a customer match: 0


# **Dataset and Columns**

In [ ]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df.columns.tolist())


CUSTOMERS
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

GEOLOCATION
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

ORDER_ITEMS
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

ORDER_PAYMENTS
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

ORDER_REVIEWS
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

ORDERS
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

PRODUCTS
['product_id', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_heigh

# **Dimcustomer**

In [ ]:
# Build DimCustomer with only the stable customer identifier

dim_customer = (
    customers[['customer_unique_id']]
    .drop_duplicates()
    .copy()
)

print("DimCustomer shape:", dim_customer.shape)
print("Duplicate keys:",
      dim_customer['customer_unique_id'].duplicated().sum())
print("Missing keys:",
      dim_customer['customer_unique_id'].isna().sum())

dim_customer.head()

DimCustomer shape: (96096, 1)
Duplicate keys: 0
Missing keys: 0


,customer_unique_id
0,861eff4711a542e4b93843c6dd7febb0
1,290c77bc529b7ac935b93aa66c333dc3
2,060e732b5b29e8181a18229c7b0b2b5e
3,259dac757896d24d7702b9acbbff3f3c
4,345ecd01c38d18a9036ed96c73b8d066


In [ ]:
# Check address variations for each real customer

address_variations = (
    customers.groupby('customer_unique_id')
    .agg(
        zip_codes=('customer_zip_code_prefix', 'nunique'),
        cities=('customer_city', 'nunique'),
        states=('customer_state', 'nunique')
    )
)

print("Customers with multiple postal codes:",
      (address_variations['zip_codes'] > 1).sum())

print("Customers with multiple cities:",
      (address_variations['cities'] > 1).sum())

print("Customers with multiple states:",
      (address_variations['states'] > 1).sum())

Customers with multiple postal codes: 250
Customers with multiple cities: 122
Customers with multiple states: 39


# **Dim Products**

In [ ]:
# Create the product dimension

dim_product = products[
    [
        'product_id',
        'product_category_name',
        'product_category_name_english',
        'product_name_length',
        'product_description_length',
        'product_photos_qty',
        'product_weight_g',
        'product_length_cm',
        'product_height_cm',
        'product_width_cm'
    ]
].drop_duplicates(subset='product_id').copy()


# Validate the product dimension

print("DimProduct shape:", dim_product.shape)

print("\nDuplicate product keys:",
      dim_product['product_id'].duplicated().sum())

print("\nMissing product keys:",
      dim_product['product_id'].isna().sum())

print("\nUnique product categories:",
      dim_product['product_category_name_english'].nunique())

# Display sample records
dim_product.head()

DimProduct shape: (32951, 10)

Duplicate product keys: 0

Missing product keys: 0

Unique product categories: 74


,product_id,product_category_name,product_category_name_english,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,art,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,baby,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [ ]:
# Count missing category values

print("Missing Portuguese categories:",
      dim_product['product_category_name'].isna().sum())

print("Missing English categories:",
      dim_product['product_category_name_english'].isna().sum())


# Find English categories without a Portuguese category

missing_portuguese = dim_product[
    dim_product['product_category_name'].isna()
]

print("\nProducts with missing Portuguese categories:",
      len(missing_portuguese))


# Show category combinations

category_combinations = (
    dim_product[
        ['product_category_name', 'product_category_name_english']
    ]
    .drop_duplicates()
    .sort_values('product_category_name_english')
)

category_combinations.head(20)

Missing Portuguese categories: 0
Missing English categories: 0

Products with missing Portuguese categories: 0


,product_category_name,product_category_name_english
79,agro_industria_e_comercio,agro_industry_and_commerce
123,climatizacao,air_conditioning
1,artes,art
1614,artes_e_artesanato,arts_and_craftmanship
591,audio,audio
36,automotivo,auto
3,bebes,baby
10,cama_mesa_banho,bed_bath_table
176,livros_interesse_geral,books_general_interest
881,livros_importados,books_imported


In [ ]:
# Get unique categories from the translation table
translation_categories = set(
    category_translation['product_category_name'].dropna()
)

# Get unique Portuguese categories from products
product_categories = set(
    dim_product['product_category_name'].dropna()
)

# Compare the two sets
print("Categories in products but not translation table:")
print(sorted(product_categories - translation_categories))

print("\nCategories in translation table but not products:")
print(sorted(translation_categories - product_categories))

print("\nPortuguese categories in products:",
      len(product_categories))

print("Portuguese categories in translation table:",
      len(translation_categories))

Categories in products but not translation table:
['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos', 'unknown']

Categories in translation table but not products:
[]

Portuguese categories in products: 74
Portuguese categories in translation table: 71


In [ ]:
# Improve English category labels for the two untranslated categories

category_label_updates = {
    'pc_gamer': 'gaming_pcs',
    'portateis_cozinha_e_preparadores_de_alimentos':
        'kitchen_appliances_and_food_processors'
}

dim_product['product_category_name_english'] = (
    dim_product['product_category_name_english']
    .replace(category_label_updates)
)

# Verify the updates

print(
    dim_product[
        dim_product['product_category_name'].isin(
            ['pc_gamer',
             'portateis_cozinha_e_preparadores_de_alimentos',
             'unknown']
        )
    ][
        ['product_category_name',
         'product_category_name_english']
    ].drop_duplicates().to_string(index=False)
)

                        product_category_name          product_category_name_english
                                      unknown                                unknown
                                     pc_gamer                             gaming_pcs
portateis_cozinha_e_preparadores_de_alimentos kitchen_appliances_and_food_processors


# **Dim Sellers**

In [ ]:
dim_seller = sellers[
    [
        'seller_id',
        'seller_zip_code_prefix',
        'seller_city',
        'seller_state'
    ]
].drop_duplicates(subset='seller_id').copy()

print("DimSeller shape:", dim_seller.shape)
print("Duplicate seller keys:", dim_seller['seller_id'].duplicated().sum())
print("Missing seller keys:", dim_seller['seller_id'].isna().sum())

dim_seller.head()

DimSeller shape: (3095, 4)
Duplicate seller keys: 0
Missing seller keys: 0


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


# **Dim Date**

In [ ]:
# Convert order date columns into datetime format

date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

# Inspect the purchase date range

print("Earliest purchase date:",
      orders['order_purchase_timestamp'].min())

print("Latest purchase date:",
      orders['order_purchase_timestamp'].max())

Earliest purchase date: 2016-09-04 21:15:19
Latest purchase date: 2018-10-17 17:30:18


In [ ]:
# Generate a complete daily calendar from 2016 through 2018

date_range = pd.date_range(
    start='2016-01-01',
    end='2018-12-31',
    freq='D'
)

dim_date = pd.DataFrame({
    'date': date_range
})

# Create useful calendar attributes

dim_date['year'] = dim_date['date'].dt.year
dim_date['month_number'] = dim_date['date'].dt.month
dim_date['month_name'] = dim_date['date'].dt.month_name()
dim_date['quarter'] = dim_date['date'].dt.quarter
dim_date['day'] = dim_date['date'].dt.day
dim_date['day_of_week'] = dim_date['date'].dt.day_name()
dim_date['day_of_week_number'] = dim_date['date'].dt.dayofweek + 1

# Inspect the dimension

print("DimDate shape:", dim_date.shape)
print("Duplicate dates:", dim_date['date'].duplicated().sum())
print("Missing dates:", dim_date['date'].isna().sum())

dim_date.head()

DimDate shape: (1096, 8)
Duplicate dates: 0
Missing dates: 0


,date,year,month_number,month_name,quarter,day,day_of_week,day_of_week_number
0,2016-01-01,2016,1,January,1,1,Friday,5
1,2016-01-02,2016,1,January,1,2,Saturday,6
2,2016-01-03,2016,1,January,1,3,Sunday,7
3,2016-01-04,2016,1,January,1,4,Monday,1
4,2016-01-05,2016,1,January,1,5,Tuesday,2


# **Final Dimension Check**

In [ ]:
dimension_checks = {
    'DimCustomer': (dim_customer, 'customer_unique_id'),
    'DimProduct': (dim_product, 'product_id'),
    'DimSeller': (dim_seller, 'seller_id'),
    'DimDate': (dim_date, 'date')
}

for name, (df, key) in dimension_checks.items():
    print(f"\n{name}")
    print("Rows:", len(df))
    print("Duplicate primary keys:", df[key].duplicated().sum())
    print("Missing primary keys:", df[key].isna().sum())


DimCustomer
Rows: 96096
Duplicate primary keys: 0
Missing primary keys: 0

DimProduct
Rows: 32951
Duplicate primary keys: 0
Missing primary keys: 0

DimSeller
Rows: 3095
Duplicate primary keys: 0
Missing primary keys: 0

DimDate
Rows: 1096
Duplicate primary keys: 0
Missing primary keys: 0


# **Fact Orders**

In [ ]:
# Start with the orders dataset
fact_orders = orders.copy()

# Add the real customer identifier
fact_orders = fact_orders.merge(
    customers[
        [
            'customer_id',
            'customer_unique_id',
            'customer_zip_code_prefix',
            'customer_city',
            'customer_state'
        ]
    ],
    on='customer_id',
    how='left',
    validate='one_to_one'
)

# Add a count column for order analysis
fact_orders['order_count'] = 1

# Inspect the resulting fact table
print("FactOrders shape:", fact_orders.shape)
print("Unique order IDs:", fact_orders['order_id'].nunique())
print("Duplicate order IDs:", fact_orders['order_id'].duplicated().sum())
print("Missing customer unique IDs:",
      fact_orders['customer_unique_id'].isna().sum())

fact_orders.head()

FactOrders shape: (99441, 13)
Unique order IDs: 99441
Duplicate order IDs: 0
Missing customer unique IDs: 0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_count
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1


# **Fact order_items**

In [ ]:
# Start with the order items dataset
fact_order_items = order_items.copy()

# Add the real customer identifier from FactOrders
fact_order_items = fact_order_items.merge(
    fact_orders[
        ['order_id', 'customer_unique_id']
    ],
    on='order_id',
    how='left',
    validate='many_to_one'
)

# Add an item count for analysis
fact_order_items['item_count'] = 1

# Validate the resulting fact table
print("FactOrderItems shape:", fact_order_items.shape)

print(
    "Unique order-item combinations:",
    fact_order_items[['order_id', 'order_item_id']].drop_duplicates().shape[0]
)

print(
    "Duplicate order-item combinations:",
    fact_order_items.duplicated(
        subset=['order_id', 'order_item_id']
    ).sum()
)

print(
    "Missing customer unique IDs:",
    fact_order_items['customer_unique_id'].isna().sum()
)

fact_order_items.head()

FactOrderItems shape: (112650, 9)
Unique order-item combinations: 112650
Duplicate order-item combinations: 0
Missing customer unique IDs: 0


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_unique_id,item_count
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,871766c5855e863f6eccc05f988b23cb,1
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,eb28e67c4c0b83846050ddfb8a35d051,1
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,3818d81c6709e39d06b2738a8d3a2474,1
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,af861d436cfc08b2c2ddefd0ba074622,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,64b576fb70d441e8f1b2d7d446e483c5,1


# **Fact order_payments**

In [ ]:
# Start with the cleaned payments dataset
fact_payments = order_payments.copy()

# Add the real customer identifier
fact_payments = fact_payments.merge(
    fact_orders[
        ['order_id', 'customer_unique_id']
    ],
    on='order_id',
    how='left',
    validate='many_to_one'
)

# Validate the resulting fact table
print("FactPayments shape:", fact_payments.shape)

print(
    "Unique payment combinations:",
    fact_payments[
        ['order_id', 'payment_sequential']
    ].drop_duplicates().shape[0]
)

print(
    "Duplicate payment combinations:",
    fact_payments.duplicated(
        subset=['order_id', 'payment_sequential']
    ).sum()
)

print(
    "Missing customer unique IDs:",
    fact_payments['customer_unique_id'].isna().sum()
)

fact_payments.head()

FactPayments shape: (103886, 6)
Unique payment combinations: 103886
Duplicate payment combinations: 0
Missing customer unique IDs: 0


,order_id,payment_sequential,payment_type,payment_installments,payment_value,customer_unique_id
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33,708ab75d2a007f0564aedd11139c7708
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39,a8b9d3a27068454b1c98cc67d4e31e6f
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71,6f70c0b2f7552832ba46eb57b1c5651e
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78,87695ed086ebd36f20404c82d20fca87
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45,4291db0da71914754618cd789aebcd56


# **Fact order_reviews**

In [ ]:
# Start with the cleaned reviews dataset
fact_reviews = order_reviews.copy()

# Add the real customer identifier
fact_reviews = fact_reviews.merge(
    fact_orders[
        ['order_id', 'customer_unique_id']
    ],
    on='order_id',
    how='left',
    validate='many_to_one'
)

# Add a review count for analysis
fact_reviews['review_count'] = 1

# Validate the resulting fact table
print("FactReviews shape:", fact_reviews.shape)

print(
    "Unique review combinations:",
    fact_reviews[
        ['review_id', 'order_id']
    ].drop_duplicates().shape[0]
)

print(
    "Duplicate review combinations:",
    fact_reviews.duplicated(
        subset=['review_id', 'order_id']
    ).sum()
)

print(
    "Missing customer unique IDs:",
    fact_reviews['customer_unique_id'].isna().sum()
)

fact_reviews.head()

FactReviews shape: (99224, 9)
Unique review combinations: 99224
Duplicate review combinations: 0
Missing customer unique IDs: 0


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,customer_unique_id,review_count
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59,68a5590b9926689be4e10f4ae2db21a8,1
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13,64190b91b656ab8f37eb89b93dc84584,1
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24,1d47144362c14e94ccdd213e8ec277d5,1
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06,c8cf6cb6b838dc7a33ed199b825e8616,1
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53,d16000272660a1fef81482ad75ba572a,1


# **Final Fact Check**

In [ ]:
fact_checks = {
    'FactOrders': (
        fact_orders,
        ['order_id']
    ),
    'FactOrderItems': (
        fact_order_items,
        ['order_id', 'order_item_id']
    ),
    'FactPayments': (
        fact_payments,
        ['order_id', 'payment_sequential']
    ),
    'FactReviews': (
        fact_reviews,
        ['review_id', 'order_id']
    )
}

for name, (df, keys) in fact_checks.items():

    duplicate_count = df.duplicated(subset=keys).sum()
    missing_count = df[keys].isna().any(axis=1).sum()

    print(f"\n{name}")
    print("Rows:", len(df))
    print("Duplicate grain combinations:", duplicate_count)
    print("Rows with missing grain keys:", missing_count)


FactOrders
Rows: 99441
Duplicate grain combinations: 0
Rows with missing grain keys: 0

FactOrderItems
Rows: 112650
Duplicate grain combinations: 0
Rows with missing grain keys: 0

FactPayments
Rows: 103886
Duplicate grain combinations: 0
Rows with missing grain keys: 0

FactReviews
Rows: 99224
Duplicate grain combinations: 0
Rows with missing grain keys: 0


# **Check customer relationships**

In [ ]:
# Let's verify that all customer identifiers in our fact tables match DimCustomer.
# Get all unique customer IDs from the customer dimension
dimension_customer_ids = set(
    dim_customer['customer_unique_id'].dropna()
)

# Check each fact table
customer_fact_checks = {
    'FactOrders': fact_orders,
    'FactOrderItems': fact_order_items,
    'FactPayments': fact_payments,
    'FactReviews': fact_reviews
}

for name, df in customer_fact_checks.items():

    fact_customer_ids = set(
        df['customer_unique_id'].dropna()
    )

    unmatched = fact_customer_ids - dimension_customer_ids

    print(f"\n{name}")
    print("Unique customers:", len(fact_customer_ids))
    print("Unmatched customer IDs:", len(unmatched))


FactOrders
Unique customers: 96096
Unmatched customer IDs: 0

FactOrderItems
Unique customers: 95420
Unmatched customer IDs: 0

FactPayments
Unique customers: 96095
Unmatched customer IDs: 0

FactReviews
Unique customers: 95380
Unmatched customer IDs: 0


# **Validate product and seller relationships**

In [ ]:
# Create sets of valid product and seller IDs

dimension_product_ids = set(
    dim_product['product_id'].dropna()
)

dimension_seller_ids = set(
    dim_seller['seller_id'].dropna()
)

# Compare FactOrderItems against the dimensions

fact_product_ids = set(
    fact_order_items['product_id'].dropna()
)

fact_seller_ids = set(
    fact_order_items['seller_id'].dropna()
)

# Product validation

unmatched_products = fact_product_ids - dimension_product_ids

print("PRODUCT VALIDATION")
print("Unique products in FactOrderItems:", len(fact_product_ids))
print("Unmatched product IDs:", len(unmatched_products))

# Seller validation

unmatched_sellers = fact_seller_ids - dimension_seller_ids

print("\nSELLER VALIDATION")
print("Unique sellers in FactOrderItems:", len(fact_seller_ids))
print("Unmatched seller IDs:", len(unmatched_sellers))

PRODUCT VALIDATION
Unique products in FactOrderItems: 32951
Unmatched product IDs: 0

SELLER VALIDATION
Unique sellers in FactOrderItems: 3095
Unmatched seller IDs: 0


# **Validate date relationships**

In [ ]:
# Identify the order date columns
order_date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

# Create a set of dates available in DimDate
dimension_dates = set(
    pd.to_datetime(dim_date['date']).dt.date
)

# Check each order date column
for col in order_date_columns:

    fact_dates = pd.to_datetime(
        fact_orders[col],
        errors='coerce'
    ).dt.date

    valid_dates = set(fact_dates.dropna())

    unmatched_dates = valid_dates - dimension_dates

    print(f"\n{col}")
    print("Unique dates:", len(valid_dates))
    print("Unmatched dates:", len(unmatched_dates))


order_purchase_timestamp
Unique dates: 634
Unmatched dates: 0

order_approved_at
Unique dates: 611
Unmatched dates: 0

order_delivered_carrier_date
Unique dates: 547
Unmatched dates: 0

order_delivered_customer_date
Unique dates: 645
Unmatched dates: 0

order_estimated_delivery_date
Unique dates: 459
Unmatched dates: 0


# **Create the modeled folder**

In [ ]:
# Create a dedicated folder for modeled datasets

modeled_path = os.path.join(project_path, 'modeled')

os.makedirs(modeled_path, exist_ok=True)

print("Modeled folder:", modeled_path)
print("Folder exists:", os.path.exists(modeled_path))

Modeled folder: /content/drive/MyDrive/Brazilian_Ecommerce/modeled
Folder exists: True


# **Save the eight modeled tables**

In [ ]:
# Collect all eight modeled tables

modeled_tables = {
    'dim_customer': dim_customer,
    'dim_product': dim_product,
    'dim_seller': dim_seller,
    'dim_date': dim_date,
    'fact_orders': fact_orders,
    'fact_order_items': fact_order_items,
    'fact_payments': fact_payments,
    'fact_reviews': fact_reviews
}

# Save each table as a CSV file

for table_name, df in modeled_tables.items():

    file_path = os.path.join(
        modeled_path,
        f'{table_name}.csv'
    )

    df.to_csv(file_path, index=False)

    print(f"Saved: {table_name}.csv | Rows: {len(df)}")

Saved: dim_customer.csv | Rows: 96096
Saved: dim_product.csv | Rows: 32951
Saved: dim_seller.csv | Rows: 3095
Saved: dim_date.csv | Rows: 1096
Saved: fact_orders.csv | Rows: 99441
Saved: fact_order_items.csv | Rows: 112650
Saved: fact_payments.csv | Rows: 103886
Saved: fact_reviews.csv | Rows: 99224
